In [0]:
%run /Shared/insclm_capstone/NB_00_config_loader.py

[SecretScope(name=' kv-insclm-cap-11'), SecretScope(name='kv-insclm')]

[SecretMetadata(key='adls-abfss-base'),
 SecretMetadata(key='adls-account-key'),
 SecretMetadata(key='adls-account-name'),
 SecretMetadata(key='adls-audit-path'),
 SecretMetadata(key='adls-base-url'),
 SecretMetadata(key='adls-bronze-path'),
 SecretMetadata(key='adls-container-name'),
 SecretMetadata(key='adls-gold-path'),
 SecretMetadata(key='adls-raw-path'),
 SecretMetadata(key='adls-rejected-path'),
 SecretMetadata(key='adls-silver-path'),
 SecretMetadata(key='database-workspace-url'),
 SecretMetadata(key='databricks-cluster-id'),
 SecretMetadata(key='databricks-pat'),
 SecretMetadata(key='file-claim-status-updates'),
 SecretMetadata(key='file-claims'),
 SecretMetadata(key='file-customer-master'),
 SecretMetadata(key='file-policy-master'),
 SecretMetadata(key='github-pat'),
 SecretMetadata(key='github-repo-url'),
 SecretMetadata(key='sql-admin-name'),
 SecretMetadata(key='sql-admin-password'),
 SecretMetadata(key='sql-connection-string'),
 SecretMetadata(key='sql-database-name'),
 S

✅ Config loaded from Key Vault successfully.
   ADLS Account  : [REDACTED]
   Container     : [REDACTED]
   ABFSS Base    : [REDACTED]
   RAW path      : [REDACTED][REDACTED]
   BRONZE path   : [REDACTED][REDACTED]
   SILVER path   : [REDACTED][REDACTED]
   GOLD path     : [REDACTED][REDACTED]
   REJECTED path : [REDACTED][REDACTED]
   AUDIT path    : [REDACTED][REDACTED]
   SQL Server    : [REDACTED]
   SQL Database  : [REDACTED]


In [0]:
from pyspark.sql import functions as F
from datetime import datetime

print("=" * 55)
print("LOADING GOLD TABLES TO AZURE SQL")
print("=" * 55)

# JDBC connection properties
connection_properties = {
    "user":     SQL_USER,
    "password": SQL_PASSWORD,
    "driver":   "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print("✅ JDBC connection properties ready")

LOADING GOLD TABLES TO AZURE SQL
✅ JDBC connection properties ready


In [0]:
# Create reporting schema in Azure SQL
print("\n📥 Creating reporting schema if not exists...")

spark.read \
    .format("jdbc") \
    .option("url", SQL_JDBC_URL) \
    .option("query",
        "IF NOT EXISTS (SELECT 1 FROM sys.schemas "
        "WHERE name = 'reporting') "
        "EXEC('CREATE SCHEMA reporting')") \
    .option("user", SQL_USER) \
    .option("password", SQL_PASSWORD) \
    .option("driver",
        "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .load()

print("✅ reporting schema ready")


📥 Creating reporting schema if not exists...
✅ reporting schema ready


In [0]:
# Helper function to write to SQL
def write_to_sql(df, table_name, mode="overwrite"):
    start     = datetime.now()
    row_count = df.count()
    print(f"\n📥 Writing {row_count:,} rows to {table_name}...")

    df.write \
        .format("jdbc") \
        .option("url",      SQL_JDBC_URL) \
        .option("dbtable",  table_name) \
        .option("user",     SQL_USER) \
        .option("password", SQL_PASSWORD) \
        .option("driver",
            "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
        .option("batchsize", 10000) \
        .mode(mode) \
        .save()

    end = datetime.now()
    print(f"✅ {table_name} → {row_count:,} rows "
          f"in {(end-start).seconds}s")
    return row_count

print("✅ write_to_sql function ready")

✅ write_to_sql function ready


In [0]:
# Write reporting.fact_claim_summary
print("\n── Table 1: reporting.fact_claim_summary ────────")

fact_claim = spark \
    .table("gold_insclm.gold_claim_summary") \
    .withColumn("loaded_at", F.current_timestamp()) \
    .drop("_gold_loaded_at")

c1 = write_to_sql(fact_claim,
                  "reporting.fact_claim_summary")


── Table 1: reporting.fact_claim_summary ────────

📥 Writing 30 rows to reporting.fact_claim_summary...
✅ reporting.fact_claim_summary → 30 rows in 1s


In [0]:
print("\n── Table 2: reporting.fact_suspicious_claims ────")

fact_suspicious = spark \
    .table("gold_insclm.gold_suspicious_claim_summary") \
    .select(
        "claim_id", "policy_id", "customer_id",
        "customer_name", "state", "risk_category",
        "claim_amount", "coverage_amount",
        "policy_status_at_claim", "document_status",
        "claim_reason", "claim_date",
        "customer_claim_count", "suspicious_score",
        "suspicion_level", "final_status",
        "final_status_date"
    ) \
    .withColumn("loaded_at", F.current_timestamp())

c2 = write_to_sql(fact_suspicious,
                  "reporting.fact_suspicious_claims")


── Table 2: reporting.fact_suspicious_claims ────

📥 Writing 1,273 rows to reporting.fact_suspicious_claims...
✅ reporting.fact_suspicious_claims → 1,273 rows in 2s


In [0]:
print("\n── Table 3: reporting.dim_policy_history ────────")

dim_policy = spark \
    .table("silver_insclm.silver_policy_dim") \
    .select(
        "policy_sk", "policy_id", "customer_id",
        "policy_type", "coverage_amount",
        "premium_amount", "policy_status",
        "start_date", "end_date",
        "effective_date", "expiry_date",
        "is_current", "record_hash"
    ) \
    .withColumn("loaded_at", F.current_timestamp())

c3 = write_to_sql(dim_policy,
                  "reporting.dim_policy_history")


── Table 3: reporting.dim_policy_history ────────

📥 Writing 1,500 rows to reporting.dim_policy_history...
✅ reporting.dim_policy_history → 1,500 rows in 2s


In [0]:
print("\n" + "=" * 55)
print("VERIFYING ROW COUNTS IN AZURE SQL")
print("=" * 55)

def read_from_sql(query):
    return spark.read \
        .format("jdbc") \
        .option("url",      SQL_JDBC_URL) \
        .option("query",    query) \
        .option("user",     SQL_USER) \
        .option("password", SQL_PASSWORD) \
        .option("driver",
            "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
        .load()

v1 = read_from_sql(
    "SELECT COUNT(*) AS cnt "
    "FROM reporting.fact_claim_summary") \
    .first()["cnt"]

v2 = read_from_sql(
    "SELECT COUNT(*) AS cnt "
    "FROM reporting.fact_suspicious_claims") \
    .first()["cnt"]

v3 = read_from_sql(
    "SELECT COUNT(*) AS cnt "
    "FROM reporting.dim_policy_history") \
    .first()["cnt"]

print(f"fact_claim_summary     : {v1:,}  (written: {c1:,})")
print(f"fact_suspicious_claims : {v2:,}  (written: {c2:,})")
print(f"dim_policy_history     : {v3:,}  (written: {c3:,})")

all_match = (v1==c1 and v2==c2 and v3==c3)
if all_match:
    print("\n✅ All counts match!")
else:
    print("\n❌ Count mismatch — investigate above")


VERIFYING ROW COUNTS IN AZURE SQL
fact_claim_summary     : 30  (written: 30)
fact_suspicious_claims : 1,273  (written: 1,273)
dim_policy_history     : 1,500  (written: 1,500)

✅ All counts match!


In [0]:
print("\n" + "=" * 55)
print("ANALYTICAL QUERY 1")
print("Approval and Rejection Rate by Policy Type")
print("=" * 55)

result1 = read_from_sql("""
    SELECT
        policy_type,
        SUM(total_claims)             AS total_claims,
        SUM(approved_claims)          AS total_approved,
        SUM(rejected_claims)          AS total_rejected,
        SUM(pending_claims)           AS total_pending,
        ROUND(AVG(approval_rate_pct), 2)
                                      AS avg_approval_rate,
        ROUND(AVG(rejection_rate_pct), 2)
                                      AS avg_rejection_rate,
        ROUND(SUM(total_claim_amount), 2)
                                      AS total_amount
    FROM reporting.fact_claim_summary
    GROUP BY policy_type
""")

result1.orderBy("total_claims", ascending=False) \
    .show(truncate=False)


ANALYTICAL QUERY 1
Approval and Rejection Rate by Policy Type
+-----------+------------+--------------+--------------+-------------+-----------------+------------------+------------+
|policy_type|total_claims|total_approved|total_rejected|total_pending|avg_approval_rate|avg_rejection_rate|total_amount|
+-----------+------------+--------------+--------------+-------------+-----------------+------------------+------------+
|Travel     |475         |184           |74            |61           |39.02            |15.68             |335312013.91|
|Motor      |417         |164           |64            |44           |39.86            |15.76             |280895744.84|
|Home       |410         |153           |58            |58           |37.47            |14.1              |295213377.19|
|Life       |400         |154           |71            |44           |38.52            |17.57             |261374349.78|
|Health     |378         |153           |52            |37           |40.23            |13

In [0]:
print("\n" + "=" * 55)
print("ANALYTICAL QUERY 2")
print("Top 10 High-Risk States by Suspicious Claims")
print("=" * 55)

result2 = read_from_sql("""
    SELECT TOP 10
        state,
        COUNT(*)                     AS total_suspicious,
        COUNT(DISTINCT customer_id)  AS unique_customers,
        ROUND(SUM(claim_amount), 2)  AS total_amount,
        ROUND(AVG(CAST(suspicious_score AS FLOAT)), 2)
                                     AS avg_score,
        SUM(CASE WHEN suspicion_level = 'HIGH'
                 THEN 1 ELSE 0 END)  AS high_risk,
        SUM(CASE WHEN suspicion_level = 'MEDIUM'
                 THEN 1 ELSE 0 END)  AS medium_risk,
        SUM(CASE WHEN suspicion_level = 'LOW'
                 THEN 1 ELSE 0 END)  AS low_risk
    FROM reporting.fact_suspicious_claims
    GROUP BY state
    ORDER BY total_suspicious DESC
""")

result2.show(truncate=False)


ANALYTICAL QUERY 2
Top 10 High-Risk States by Suspicious Claims
+--------------+----------------+----------------+------------+---------+---------+-----------+--------+
|state         |total_suspicious|unique_customers|total_amount|avg_score|high_risk|medium_risk|low_risk|
+--------------+----------------+----------------+------------+---------+---------+-----------+--------+
|Maharashtra   |312             |112             |209732119.04|1.26     |5        |70         |237     |
|Karnataka     |171             |50              |125665330.13|1.42     |11       |49         |111     |
|Rajasthan     |122             |45              |85451836.68 |1.18     |1        |20         |101     |
|Madhya Pradesh|119             |46              |80176068.83 |1.31     |2        |33         |84      |
|Uttar Pradesh |109             |40              |77241181.75 |1.38     |6        |29         |74      |
|Telangana     |102             |34              |71190031.27 |1.26     |1        |25         |

In [0]:
print("\n" + "=" * 55)
print("ANALYTICAL QUERY 3")
print("Top 10 Customers by Suspicious Claim Amount")
print("=" * 55)

result3 = read_from_sql("""
    SELECT TOP 10
        customer_id,
        customer_name,
        state,
        risk_category,
        COUNT(DISTINCT claim_id)        AS total_suspicious_claims,
        ROUND(SUM(claim_amount), 2)     AS total_suspicious_amount,
        MAX(customer_claim_count)       AS overall_claim_count,
        MAX(suspicious_score)           AS max_suspicious_score,
        SUM(CASE WHEN suspicion_level = 'HIGH'
                 THEN 1 ELSE 0 END)     AS high_risk_claims
    FROM reporting.fact_suspicious_claims
    GROUP BY
        customer_id,
        customer_name,
        state,
        risk_category
    ORDER BY total_suspicious_amount DESC
""")

result3.show(truncate=False)


ANALYTICAL QUERY 3
Top 10 Customers by Suspicious Claim Amount
+-----------+-------------+-------------+-------------+-----------------------+-----------------------+-------------------+--------------------+----------------+
|customer_id|customer_name|state        |risk_category|total_suspicious_claims|total_suspicious_amount|overall_claim_count|max_suspicious_score|high_risk_claims|
+-----------+-------------+-------------+-------------+-----------------------+-----------------------+-------------------+--------------------+----------------+
|ICUST00179 |Vivaan Chopra|West Bengal  |High         |8                      |8366815.80             |8                  |2                   |0               |
|ICUST00948 |Amit Chopra  |Delhi        |High         |10                     |8342159.74             |10                 |2                   |0               |
|ICUST00261 |Ishaan Das   |Maharashtra  |Medium       |10                     |8226479.62             |10                 |2  

In [0]:
print("\n" + "=" * 55)
print("AZURE SQL REPORTING SUMMARY")
print("=" * 55)
print(f"fact_claim_summary     : {c1:,} rows ✅")
print(f"fact_suspicious_claims : {c2:,} rows ✅")
print(f"dim_policy_history     : {c3:,} rows ✅")
print("\n3 Analytical Queries executed ✅")
print("=" * 55)
print("✅ NB_09 complete.")
print("   Next: Run NB_10_audit_logger")


AZURE SQL REPORTING SUMMARY
fact_claim_summary     : 30 rows ✅
fact_suspicious_claims : 1,273 rows ✅
dim_policy_history     : 1,500 rows ✅

3 Analytical Queries executed ✅
✅ NB_09 complete.
   Next: Run NB_10_audit_logger
